# EduSense — Ottawa Traffic Collisions
## Module A · Détection de hotspots géographiques (DBSCAN)

**Objectif :** identifier les zones géographiques denses en collisions à l'aide de DBSCAN,  
puis profiler chaque cluster selon le volume, la gravité et les usagers impliqués.

**Notebook autonome** — repart de `Traffic_Collision_Data.csv`.

---
### Plan
1. Preprocessing  
2. Rappel DBSCAN + choix des paramètres (k-distance plot interactif)  
3. Exécution DBSCAN  
4. Carte des clusters (widget interactif)  
5. Profil détaillé d'un cluster (widget sélecteur)  
6. Comparaison des clusters — métriques clés  
7. Synthèse

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

PALETTE = {
    'fatal':    '#C0392B',
    'severe':   '#E67E22',
    'blessure': '#F1C40F',
    'materiel': '#3498DB',
    'hiver':    '#8E44AD',
    'vuln':     '#27AE60',
    'neutral':  '#95A5A6',
    'noise':    '#D5D8DC',
}
print("Imports OK")

Imports OK


### 1 · Preprocessing

In [2]:
import pandas as pd
import numpy as np
from pyproj import Transformer
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('Traffic_Collision_Data.csv')
raw_count = len(df)

df = df[df['Lat'] != 0.0].copy()
df = df[df['Lat'].between(44.9, 45.6) & df['Long'].between(-76.4, -75.2)].copy()

transformer = Transformer.from_crs("EPSG:4326", "EPSG:32618", always_xy=True)
df['utm_x'], df['utm_y'] = transformer.transform(df['Long'].values, df['Lat'].values)

df['Accident_Date'] = pd.to_datetime(df['Accident_Date'], format='%m/%d/%Y', errors='coerce')
df = df[df['Accident_Date'].notna()].copy()
df['month']       = df['Accident_Date'].dt.month
df['day_of_week'] = df['Accident_Date'].dt.dayofweek
df['saison']      = df['month'].map({
    12:'Hiver',1:'Hiver',2:'Hiver',
    3:'Printemps',4:'Printemps',5:'Printemps',
    6:'Été',7:'Été',8:'Été',
    9:'Automne',10:'Automne',11:'Automne'})
df['is_weekend']  = df['day_of_week'].isin([5,6]).astype(int)

def parse_coded(series):
    code  = series.str.extract(r'^(\d+)')[0].astype(float)
    label = series.str.extract(r'^\d+\s*-\s*(.+)$')[0].str.strip()
    return code, label

for orig, prefix in {
    'Classification_Of_Accident':'classif',
    'Initial_Impact_Type':'impact',
    'Road_1_Surface_Condition':'surface',
    'Environment_Condition_1':'meteo',
    'Light':'lumiere',
    'Traffic_Control':'controle',
}.items():
    c, l = parse_coded(df[orig].fillna('00 - Unknown'))
    df[f'{prefix}_code'], df[f'{prefix}_label'] = c, l
    for suf in ['_label','_code']:
        col = f'{prefix}{suf}'
        if df[col].isna().any():
            df[col] = df[col].fillna(df[col].mode()[0])

df['severity_score'] = df['Classification_Of_Accident'].map({
    '03 - P.D. only':0,'04 - Non-reportable':0,
    '02 - Non-fatal injury':1,'01 - Fatal injury':2
}).fillna(0).astype(int)
df['injury_rank'] = df['Max_injury'].map({'Minimal':1,'Minor':2,'Major':3,'Fatal':4})
df['is_severe']   = ((df['severity_score']==2)|(df['injury_rank'].isin([3,4])).fillna(False)).astype(int)

for col in ['num_of_pedestrians','num_of_bicycles','num_of_motorcycles']:
    df[col] = df[col].fillna(0).astype(int)
df['num_of_vehicles']  = df['num_of_vehicles'].fillna(df['num_of_vehicles'].median()).round().astype(int)
df['has_pedestrian']   = (df['num_of_pedestrians']>0).astype(int)
df['has_bicycle']      = (df['num_of_bicycles']>0).astype(int)
df['has_motorcycle']   = (df['num_of_motorcycles']>0).astype(int)
df['has_vulnerable']   = ((df['has_pedestrian']|df['has_bicycle']|df['has_motorcycle'])>0).astype(int)

df['is_winter_conditions'] = (
    df['surface_label'].isin(['Loose snow','Packed snow','Ice','Slush']) |
    df['meteo_label'].isin(['Snow','Freezing Rain','Drifting Snow'])
).astype(int)
df['is_low_light']  = df['lumiere_label'].isin(['Dark','Dusk','Dawn']).astype(int)
df['has_signal']    = (df['controle_label']=='Traffic signal').astype(int)
df['no_control']    = (df['controle_label']=='No control').astype(int)
df['impact_simple'] = df['impact_label'].map({
    'Rear end':'arriere','Angle':'angle','Sideswipe':'lateral',
    'Turning movement':'virage','SMV other':'smv',
    'SMV unattended vehicle':'smv','Approaching':'face_a_face','Other':'autre'
}).fillna('autre')

print(f"Dataset prêt : {len(df):,} lignes × {df.shape[1]} colonnes")

Dataset prêt : 94,334 lignes × 58 colonnes


### 2 · Rappel DBSCAN & calibration des paramètres

DBSCAN classe chaque point comme :
- **core point** — a au moins `min_samples` voisins dans un rayon `eps`
- **border point** — dans le rayon d'un core point, mais pas core lui-même
- **noise** (label = -1) — isolé, n'appartient à aucun cluster

**Paramètres clés :**
- `eps` (mètres) — rayon de voisinage. Travail en UTM → unité directement en mètres
- `min_samples` — nombre minimal de collisions pour former un core point

**Calibration par k-distance plot :** on trie les distances au k-ième plus proche voisin.  
Le "coude" indique un eps naturel.

In [3]:
coords = df[['utm_x','utm_y']].values

# ── Widget k-distance plot interactif ───────────────────────────────────────
k_slider = widgets.IntSlider(value=30, min=5, max=80, step=5,
                              description='k (min_samples candidat) :',
                              style={'description_width':'220px'},
                              layout=widgets.Layout(width='520px'))

out_kdist = widgets.Output()

def plot_kdist(k):
    with out_kdist:
        clear_output(wait=True)
        nbrs = NearestNeighbors(n_neighbors=k, n_jobs=-1).fit(coords)
        dists, _ = nbrs.kneighbors(coords)
        k_dists  = np.sort(dists[:, -1])

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(k_dists, color=PALETTE['materiel'], linewidth=1.2)
        ax.set_xlabel("Points triés par distance croissante au k-ième voisin")
        ax.set_ylabel(f"Distance au {k}e voisin (mètres)")
        ax.set_title(f"k-distance plot — k={k}  |  chercher le coude pour choisir eps")
        ax.set_facecolor('#f8f8f6')
        ax.grid(True, color='white')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        # Annoter quelques percentiles
        for p, lbl in [(50,'p50'),(90,'p90'),(95,'p95')]:
            val = np.percentile(k_dists, p)
            idx = np.searchsorted(k_dists, val)
            ax.axhline(val, linestyle='--', linewidth=0.8, color=PALETTE['severe'], alpha=0.7)
            ax.text(len(k_dists)*0.02, val+8, f"{lbl} = {val:.0f}m",
                    fontsize=8, color=PALETTE['severe'])
        plt.tight_layout()
        plt.show()

k_slider.observe(lambda c: plot_kdist(c['new']), names='value')

display(widgets.VBox([
    widgets.HTML("<b>Ajuste k pour observer comment le coude se déplace :</b>"),
    k_slider,
    out_kdist
]))
plot_kdist(k_slider.value)

### 3 · Exécution DBSCAN

In [4]:
# ── Widget paramètres DBSCAN ────────────────────────────────────────────────
eps_slider  = widgets.IntSlider(value=80, min=50, max=250, step=10,
                                 description='eps (mètres) :',
                                 style={'description_width':'160px'},
                                 layout=widgets.Layout(width='480px'))
min_slider  = widgets.IntSlider(value=20, min=5, max=80, step=5,
                                 description='min_samples :',
                                 style={'description_width':'160px'},
                                 layout=widgets.Layout(width='480px'))
run_btn     = widgets.Button(description='Lancer DBSCAN',
                              button_style='primary',
                              icon='play',
                              layout=widgets.Layout(width='200px'))
out_params  = widgets.Output()

# Stockage partagé pour les cellules suivantes
state = {}

def run_dbscan(_):
    with out_params:
        clear_output(wait=True)
        eps  = eps_slider.value
        mins = min_slider.value
        print(f"Calcul DBSCAN — eps={eps}m, min_samples={mins} ...")
        db     = DBSCAN(eps=eps, min_samples=mins, n_jobs=-1).fit(coords)
        labels = db.labels_
        df['cluster'] = labels

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise    = (labels == -1).sum()
        pct_noise  = n_noise / len(labels) * 100

        # Agréger stats par cluster
        df_c = df[df['cluster'] >= 0].copy()
        cstats = df_c.groupby('cluster').agg(
            n            = ('severity_score','count'),
            n_severe     = ('is_severe','sum'),
            n_fatal      = ('severity_score', lambda x:(x==2).sum()),
            n_blessure   = ('severity_score', lambda x:(x==1).sum()),
            n_vuln       = ('has_vulnerable','sum'),
            n_hiver      = ('is_winter_conditions','sum'),
            n_low_light  = ('is_low_light','sum'),
            pct_severe   = ('is_severe','mean'),
            pct_vuln     = ('has_vulnerable','mean'),
            pct_hiver    = ('is_winter_conditions','mean'),
            pct_low_light= ('is_low_light','mean'),
            lat          = ('Lat','mean'),
            lon          = ('Long','mean'),
            utm_x        = ('utm_x','mean'),
            utm_y        = ('utm_y','mean'),
        ).reset_index()
        cstats['score_risque'] = (
            cstats['n']          * 0.3  +
            cstats['n_blessure'] * 2    +
            cstats['n_severe']   * 5    +
            cstats['n_fatal']    * 20
        )

        state['labels']  = labels
        state['cstats']  = cstats
        state['eps']     = eps
        state['mins']    = mins

        print(f"\n{'='*50}")
        print(f"  Clusters détectés : {n_clusters}")
        print(f"  Points bruit      : {n_noise:,} ({pct_noise:.1f}%)")
        print(f"  Points clustérisés: {len(labels)-n_noise:,} ({100-pct_noise:.1f}%)")
        print(f"{'='*50}")
        print(f"  Taille min cluster : {cstats['n'].min()}")
        print(f"  Taille médiane     : {cstats['n'].median():.0f}")
        print(f"  Taille max cluster : {cstats['n'].max()}")
        print(f"\nTop 5 clusters par score de risque :")
        cols_show = ['cluster','n','n_blessure','n_severe','n_fatal','pct_vuln','score_risque']
        print(cstats.nlargest(5,'score_risque')[cols_show].to_string(index=False))

run_btn.on_click(run_dbscan)

display(widgets.VBox([
    widgets.HTML("<b>Paramètres DBSCAN :</b>"),
    eps_slider, min_slider,
    run_btn,
    out_params
]))
run_dbscan(None)  # Lancer avec valeurs par défaut

### 4 · Carte des clusters

Widget interactif : filtre par taille minimum, colore par métrique de risque,  
zoom sur les top clusters.

In [5]:
# ── Contrôles carte ─────────────────────────────────────────────────────────
min_size_slider = widgets.IntSlider(
    value=50, min=20, max=500, step=10,
    description='Taille min cluster :',
    style={'description_width':'180px'},
    layout=widgets.Layout(width='520px')
)
color_by = widgets.RadioButtons(
    options=[
        ('Volume (nb collisions)',  'n'),
        ('% collisions graves',     'pct_severe'),
        ('% usagers vulnérables',   'pct_vuln'),
        ('% conditions hivernales', 'pct_hiver'),
        ('Score de risque',         'score_risque'),
    ],
    value='score_risque',
    description='Colorier par :',
    style={'description_width':'120px'},
)
show_noise = widgets.Checkbox(value=False, description='Afficher le bruit')
top_n_slider = widgets.IntSlider(
    value=10, min=3, max=30, step=1,
    description='Annoter top N :',
    style={'description_width':'130px'},
    layout=widgets.Layout(width='400px')
)
update_btn_map = widgets.Button(description='Mettre à jour la carte',
                                 button_style='info', icon='refresh',
                                 layout=widgets.Layout(width='220px'))
out_map = widgets.Output()

def plot_map(_):
    with out_map:
        clear_output(wait=True)
        if 'cstats' not in state:
            print("Lance d'abord DBSCAN dans la section 3.")
            return

        cstats  = state['cstats']
        labels  = state['labels']
        metric  = color_by.value
        min_sz  = min_size_slider.value
        top_n   = top_n_slider.value

        cs = cstats[cstats['n'] >= min_sz].copy()
        n_shown = len(cs)

        vmin, vmax = cs[metric].min(), cs[metric].max()
        norm   = mcolors.Normalize(vmin=vmin, vmax=vmax)
        cmap   = cm.get_cmap('RdYlGn_r') if ('pct' in metric or metric == 'score_risque') else cm.get_cmap('YlOrRd')

        fig, ax = plt.subplots(figsize=(12, 9))
        ax.set_facecolor('#EAF2FB')
        ax.set_title(
            "Hotspots DBSCAN (eps={}m, min={}) — {} clusters affichés (≥{} collisions)".format(
                state['eps'], state['mins'], n_shown, min_sz),
            fontsize=12, fontweight='bold'
        )

        if show_noise.value:
            noise_mask = labels == -1
            ax.scatter(df.loc[noise_mask, 'Long'], df.loc[noise_mask, 'Lat'],
                       c=PALETTE['noise'], s=1, alpha=0.15, zorder=1, label='Bruit')

        clustered = df[df['cluster'].isin(cs['cluster'].values)]
        cluster_color_map = {
            row['cluster']: cmap(norm(row[metric]))
            for _, row in cs.iterrows()
        }
        colors_pts = [cluster_color_map.get(c, PALETTE['noise']) for c in clustered['cluster']]
        ax.scatter(clustered['Long'], clustered['Lat'],
                   c=colors_pts, s=3, alpha=0.35, zorder=2)

        size_scale = (cs['n'] - cs['n'].min()) / (cs['n'].max() - cs['n'].min() + 1)
        scatter_sizes = 80 + size_scale * 600
        sc = ax.scatter(cs['lon'], cs['lat'],
                        c=cs[metric], cmap=cmap, norm=norm,
                        s=scatter_sizes, alpha=0.85,
                        edgecolors='white', linewidths=0.8, zorder=4)

        top_clusters = cs.nlargest(top_n, metric)
        for _, row in top_clusters.iterrows():
            annot_label = "#{} - {} coll.".format(int(row['cluster']), int(row['n']))
            ax.annotate(
                annot_label,
                xy=(row['lon'], row['lat']),
                xytext=(6, 6), textcoords='offset points',
                fontsize=7, fontweight='bold',
                color='#2C3E50',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.75, edgecolor='none'),
                zorder=5
            )

        label_map = {
            'n':             'Volume (collisions)',
            'pct_severe':    '% collisions graves',
            'pct_vuln':      '% usagers vulnérables',
            'pct_hiver':     '% conditions hivernales',
            'score_risque':  'Score de risque',
        }
        cbar = plt.colorbar(sc, ax=ax, shrink=0.6, pad=0.02)
        cbar.set_label(label_map[metric], fontsize=9)

        ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.show()
        plt.close('all')
        best = cs.loc[cs['score_risque'].idxmax()]
        print("\n{} clusters affichés  |  total collisions : {:,}  |  score max : {:.0f} (cluster #{})".format(
            n_shown, cs['n'].sum(), best['score_risque'], int(best['cluster'])))

update_btn_map.on_click(plot_map)

display(widgets.VBox([
    widgets.HTML("<b>Options d'affichage :</b>"),
    widgets.HBox([
        widgets.VBox([min_size_slider, top_n_slider, show_noise]),
        widgets.VBox([color_by])
    ]),
    update_btn_map,
    out_map
]))
plot_map(None)

### 5 · Profil détaillé d'un cluster

Sélectionne un cluster pour voir sa distribution complète :  
gravité, usagers, conditions, saison, type d'impact.

In [6]:
# ── Widget sélecteur de cluster ─────────────────────────────────────────────
cluster_input = widgets.BoundedIntText(
    value=0, min=0, max=2000, step=1,
    description='N° cluster :',
    style={'description_width':'120px'},
    layout=widgets.Layout(width='250px')
)
profile_btn = widgets.Button(description='Afficher le profil',
                              button_style='success', icon='bar-chart',
                              layout=widgets.Layout(width='200px'))
out_profile = widgets.Output()

def plot_profile(_):
    with out_profile:
        clear_output(wait=True)
        if 'cstats' not in state:
            print("Lance d'abord DBSCAN dans la section 3.")
            return

        cid     = cluster_input.value
        cstats  = state['cstats']

        if cid not in cstats['cluster'].values:
            print(f"Cluster #{cid} non trouvé. Clusters disponibles : {sorted(cstats['cluster'].values[:20].tolist())} ...")
            return

        sub  = df[df['cluster'] == cid].copy()
        row  = cstats[cstats['cluster']==cid].iloc[0]

        fig  = plt.figure(figsize=(16, 9))
        fig.suptitle(
            f"Profil — Cluster #{cid}  |  {int(row['n']):,} collisions  |  "
            f"lat={row['lat']:.4f}, lon={row['lon']:.4f}  |  score risque={row['score_risque']:.0f}",
            fontsize=12, fontweight='bold'
        )

        gs = fig.add_gridspec(2, 4, hspace=0.45, wspace=0.40)

        # A — Gravité
        ax = fig.add_subplot(gs[0, 0])
        sev = sub['severity_score'].value_counts().sort_index()
        labels_sev = {0:'Matériel/NR', 1:'Blessure', 2:'Fatal'}
        colors_sev = [PALETTE['materiel'], PALETTE['blessure'], PALETTE['fatal']]
        ax.bar([labels_sev[i] for i in sev.index], sev.values,
               color=[colors_sev[i] for i in sev.index], edgecolor='white')
        ax.set_title("Gravité"); ax.set_ylabel("Collisions")
        ax.set_facecolor('#f8f8f6')
        for i, v in enumerate(sev.values):
            ax.text(i, v+0.5, str(v), ha='center', fontsize=8, fontweight='bold')

        # B — Saison
        ax = fig.add_subplot(gs[0, 1])
        saison_c = sub['saison'].value_counts()
        colors_s = {'Hiver':'#8E44AD','Printemps':'#27AE60','Été':'#F39C12','Automne':'#E74C3C'}
        ax.bar(saison_c.index, saison_c.values,
               color=[colors_s.get(s, PALETTE['neutral']) for s in saison_c.index],
               edgecolor='white')
        ax.set_title("Saison"); ax.set_ylabel("Collisions")
        ax.set_facecolor('#f8f8f6')
        ax.tick_params(axis='x', rotation=20)

        # C — Éclairage
        ax = fig.add_subplot(gs[0, 2])
        light_c = sub['lumiere_label'].value_counts()
        ax.barh(light_c.index[::-1], light_c.values[::-1],
                color=PALETTE['materiel'], edgecolor='white')
        ax.set_title("Éclairage"); ax.set_xlabel("Collisions")
        ax.set_facecolor('#f8f8f6')

        # D — Type d'impact
        ax = fig.add_subplot(gs[0, 3])
        imp_c = sub['impact_simple'].value_counts()
        ax.barh(imp_c.index[::-1], imp_c.values[::-1],
                color=PALETTE['severe'], edgecolor='white')
        ax.set_title("Type d'impact"); ax.set_xlabel("Collisions")
        ax.set_facecolor('#f8f8f6')

        # E — Surface de chaussée
        ax = fig.add_subplot(gs[1, 0])
        surf_c = sub['surface_label'].value_counts().head(6)
        colors_surf = [PALETTE['hiver'] if any(w in s for w in ['snow','Snow','Ice','Slush','ice'])
                       else PALETTE['materiel'] for s in surf_c.index]
        ax.barh(surf_c.index[::-1], surf_c.values[::-1],
                color=colors_surf[::-1], edgecolor='white')
        ax.set_title("Surface (violet=hivernal)"); ax.set_xlabel("Collisions")
        ax.set_facecolor('#f8f8f6')

        # F — Contrôle de circulation
        ax = fig.add_subplot(gs[1, 1])
        ctrl_c = sub['controle_label'].value_counts().head(5)
        ax.barh(ctrl_c.index[::-1], ctrl_c.values[::-1],
                color=PALETTE['neutral'], edgecolor='white')
        ax.set_title("Contrôle de circulation"); ax.set_xlabel("Collisions")
        ax.set_facecolor('#f8f8f6')

        # G — Usagers vulnérables
        ax = fig.add_subplot(gs[1, 2])
        vuln_data = {
            'Piéton':   int(sub['has_pedestrian'].sum()),
            'Cycliste': int(sub['has_bicycle'].sum()),
            'Moto':     int(sub['has_motorcycle'].sum()),
            'Aucun':    int((sub['has_vulnerable']==0).sum()),
        }
        colors_v = [PALETTE['vuln'], PALETTE['fatal'], PALETTE['severe'], PALETTE['neutral']]
        ax.bar(list(vuln_data.keys()), list(vuln_data.values()),
               color=colors_v, edgecolor='white')
        ax.set_title("Usagers impliqués"); ax.set_ylabel("Collisions")
        ax.set_facecolor('#f8f8f6')
        for i, v in enumerate(vuln_data.values()):
            ax.text(i, v+0.5, str(v), ha='center', fontsize=8)

        # H — Carte locale du cluster (scatter lat/lon)
        ax = fig.add_subplot(gs[1, 3])
        color_sev_pts = sub['severity_score'].map(
            {0:PALETTE['materiel'], 1:PALETTE['blessure'], 2:PALETTE['fatal']}
        )
        ax.scatter(sub['Long'], sub['Lat'],
                   c=color_sev_pts, s=15, alpha=0.5, edgecolors='none')
        ax.scatter(row['lon'], row['lat'],
                   c='black', s=100, marker='+', linewidths=2, zorder=5, label='Centroïde')
        ax.set_title("Emprise géographique")
        ax.set_xlabel("Lon"); ax.set_ylabel("Lat")
        ax.set_facecolor('#EAF2FB')
        patches = [mpatches.Patch(color=PALETTE['materiel'], label='Matériel'),
                   mpatches.Patch(color=PALETTE['blessure'], label='Blessure'),
                   mpatches.Patch(color=PALETTE['fatal'],    label='Fatal')]
        ax.legend(handles=patches, frameon=False, fontsize=7)

        plt.tight_layout()
        plt.show()
        plt.close('all')

        # Tableau récapitulatif
        print(f"\n{'─'*55}")
        print(f"  Cluster #{cid} — résumé")
        print(f"{'─'*55}")
        print(f"  Collisions totales     : {int(row['n']):,}")
        print(f"  dont graves            : {int(row['n_severe'])} ({row['pct_severe']*100:.1f}%)")
        print(f"  dont fatales           : {int(row['n_fatal'])}")
        print(f"  Usagers vulnérables    : {int(row['n_vuln'])} ({row['pct_vuln']*100:.1f}%)")
        print(f"  Conditions hivernales  : {int(row['n_hiver'])} ({row['pct_hiver']*100:.1f}%)")
        print(f"  Basse luminosité       : {int(row['n_low_light'])} ({row['pct_low_light']*100:.1f}%)")
        print(f"  Score de risque        : {row['score_risque']:.0f}")

profile_btn.on_click(plot_profile)

display(widgets.VBox([
    widgets.HTML("<b>Sélectionne un cluster (voir la carte ci-dessus pour les numéros) :</b>"),
    widgets.HBox([cluster_input, profile_btn]),
    out_profile
]))
plot_profile(None)

### 6 · Comparaison des top clusters — métriques clés

In [7]:
# ── Widget comparaison ──────────────────────────────────────────────────────
top_n_comp = widgets.IntSlider(
    value=15, min=5, max=40, step=1,
    description='Top N clusters :',
    style={'description_width':'150px'},
    layout=widgets.Layout(width='440px')
)
sort_by_comp = widgets.Dropdown(
    options=[
        ('Score de risque',         'score_risque'),
        ('Volume',                  'n'),
        ('% graves',                'pct_severe'),
        ('% vulnérables',           'pct_vuln'),
        ('% conditions hivernales', 'pct_hiver'),
    ],
    value='score_risque',
    description='Trier par :',
    style={'description_width':'100px'},
    layout=widgets.Layout(width='360px')
)
update_btn_comp = widgets.Button(description='Mettre à jour',
                                  button_style='info', icon='refresh',
                                  layout=widgets.Layout(width='180px'))
out_comp = widgets.Output()

def plot_comparison(_):
    with out_comp:
        clear_output(wait=True)
        if 'cstats' not in state:
            print("Lance d'abord DBSCAN dans la section 3.")
            return

        cstats = state['cstats']
        top_n  = top_n_comp.value
        sortby = sort_by_comp.value

        top = cstats.nlargest(top_n, sortby).copy()
        top['label'] = top['cluster'].apply(lambda x: f"#{int(x)}")

        fig, axes = plt.subplots(1, 4, figsize=(18, max(5, top_n * 0.35 + 1)))
        fig.suptitle(
            f"Comparaison des top {top_n} clusters — triés par {sort_by_comp.label}",
            fontsize=12, fontweight='bold'
        )

        metrics = [
            ('n',           'Volume (collisions)',    PALETTE['materiel']),
            ('pct_severe',  '% collisions graves',   PALETTE['fatal']),
            ('pct_vuln',    '% usagers vulnérables', PALETTE['vuln']),
            ('score_risque','Score de risque',        PALETTE['severe']),
        ]

        for ax, (col, title, color) in zip(axes, metrics):
            vals  = top[col].values
            ypos  = range(len(top))
            # Normaliser pour la couleur
            if vals.max() > vals.min():
                norm_vals = (vals - vals.min()) / (vals.max() - vals.min())
            else:
                norm_vals = np.ones(len(vals))
            bar_colors = [plt.cm.YlOrRd(0.3 + 0.6 * v) for v in norm_vals]

            ax.barh(list(ypos), vals, color=bar_colors, edgecolor='white')
            ax.set_yticks(list(ypos))
            ax.set_yticklabels(top['label'].values, fontsize=8)
            ax.set_title(title, fontsize=9, fontweight='bold')
            ax.set_facecolor('#f8f8f6')
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
            ax.invert_yaxis()

            # Annoter les valeurs
            for i, v in enumerate(vals):
                fmt = f"{v:.1%}" if 'pct' in col else f"{v:.0f}"
                ax.text(v * 1.01, i, fmt, va='center', fontsize=7)

        plt.tight_layout()
        plt.show()
        plt.close('all')

update_btn_comp.on_click(plot_comparison)

display(widgets.VBox([
    widgets.HBox([top_n_comp, sort_by_comp, update_btn_comp]),
    out_comp
]))
plot_comparison(None)

### 7 · Synthèse

| Élément | Valeur par défaut (eps=80m, min=20) |
|---------|--------------------------------------|
| Clusters détectés | ~626 |
| Points bruit | ~32% |
| Taille médiane cluster | ~47 collisions |
| Cluster le plus dense | Centre-ville d'Ottawa |

**Recommandations pour la suite :**
- Les clusters à **score de risque élevé + % vulnérables élevé** sont les cibles prioritaires pour le module E
- Les clusters à **% hivernal élevé** alimentent le module D (segmentation environnement)
- Les intersections dans les top clusters à gravité élevée alimentent le module F (anomalies)

**Paramètre eps :** une valeur de 80-100m correspond à la largeur d'un carrefour standard à Ottawa, ce qui est sémantiquement cohérent — un "hotspot" est alors une zone d'un ou deux carrefours consécutifs.